# Gold Price Forecasting

## Project Overview

Forecasts the **daily gold price** (USD per troy ounce) over a 14-day horizon. Synthetic data spans 2 years with a gradual uptrend, volatility clustering, and safe-haven demand spikes.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily gold prices, predict the next 14 days. Gold price forecasts inform investment allocation, central bank reserves management, jewelry industry planning, and inflation hedging strategies.

## Why This Project Matters

Gold is the world's primary safe-haven asset. Its price reflects inflation expectations, interest rates, geopolitical risk, and USD strength. Accurate short-term forecasts help portfolio managers, central banks, and gold-dependent industries manage risk and timing.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily gold prices
- Gradual uptrend (inflation/demand growth)
- Volatility clustering (calm and volatile regimes)
- Safe-haven spikes during stress events
- Mild weekly pattern (trading days)

| Column | Description |
|--------|-------------|
| `date` | Date |
| `gold_usd_oz` | Daily gold price (USD per troy ounce) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'gold_usd_oz'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Gradual uptrend
trend = 1800 + 0.3 * t

# Volatility clustering (regime switching)
vol = np.ones(N_POINTS) * 8
for s in range(0, N_POINTS, 180):
    high_vol_start = min(s + np.random.randint(60, 150), N_POINTS - 30)
    duration = min(np.random.randint(15, 40), N_POINTS - high_vol_start)
    vol[high_vol_start:high_vol_start + duration] = 20

# Random walk with drift and varying volatility
price = np.zeros(N_POINTS)
price[0] = trend[0]
for i in range(1, N_POINTS):
    price[i] = price[i-1] + 0.3 + vol[i] * np.random.normal() * 0.15

# Safe-haven spikes
for s in range(0, N_POINTS, 150):
    spike_day = min(s + np.random.randint(40, 120), N_POINTS - 5)
    spike = np.random.uniform(30, 80)
    for j in range(min(5, N_POINTS - spike_day)):
        price[spike_day + j] += spike * np.exp(-0.4 * j)

gold_usd_oz = np.maximum(price, 1500).round(2)

df = pd.DataFrame({'date': dates, 'gold_usd_oz': gold_usd_oz})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,gold_usd_oz
0,2022-01-01,1800.00
1,2022-01-02,1802.20
2,2022-01-03,1803.42
3,2022-01-04,1803.15
4,2022-01-05,1804.10


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('gold_usd_oz Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("gold_usd_oz Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

gold_usd_oz Statistics:
count     730.000000
mean     1895.032767
std        56.477858
min      1796.700000
25%      1844.032500
50%      1909.600000
75%      1939.332500
max      2043.090000
Name: gold_usd_oz, dtype: float64

CV: 0.030
Skewness: -0.302


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        5.1 | RMSE:        5.5 | MAPE:  0.26%
Seasonal Naive (7)             | MAE:        3.4 | RMSE:        4.0 | MAPE:  0.17%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                          Adjusted R-Squared      R-Squared         RMSE  Time Taken
Model                                                                               
KernelRidge                     4.189604e+06 -322276.127200  1896.134601    0.020697
GaussianProcessRegressor        2.865239e+06 -220401.914321  1568.061486    0.035586
MLPRegressor                    2.220234e+06 -170786.169235  1380.327186    0.236474
LinearSVR                       1.976472e+06 -152035.218836  1302.350712    0.006501
DummyRegressor                  7.604266e+03    -583.866614    80.776092    0.005679
QuantileRegressor               5.183717e+03    -397.670570    66.690122    0.033139
NuSVR                           3.105430e+03    -237.802277    51.614729    0.015328
SVR                             2.886964e+03    -220.997197    49.765477    0.020204
ExtraTreeRegressor              8.070016e+02     -61.000121    26.299694    0.006938
AdaBoostRegressor               2.949777e+02

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: lgbm
FLAML (lgbm)                   | MAE:        2.1 | RMSE:        3.0 | MAPE:  0.11%
FLAML Test (lgbm)              | MAE:        5.1 | RMSE:        5.5 | MAPE:  0.26%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        7.9 | RMSE:        8.4 | MAPE:  0.40%
SF AutoETS                     | MAE:        6.5 | RMSE:        6.9 | MAPE:  0.33%
SF SeasonalNaive               | MAE:        5.2 | RMSE:        5.9 | MAPE:  0.27%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE     MAPE
      FLAML (lgbm) 2.097913 3.006432 0.106151
Seasonal Naive (7) 3.433571 4.042456 0.174000
 FLAML Test (lgbm) 5.086205 5.493468 0.257770
Naive (Last Value) 5.142857 5.532199 0.260415
  SF SeasonalNaive 5.231429 5.911084 0.265149
        SF AutoETS 6.519706 6.863075 0.330394
      SF AutoARIMA 7.919209 8.389980 0.401320

Top 3:
             model      MAE     RMSE     MAPE
      FLAML (lgbm) 2.097913 3.006432 0.106151
Seasonal Naive (7) 3.433571 4.042456 0.174000
 FLAML Test (lgbm) 5.086205 5.493468 0.257770


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -5.09, Std: 2.08


## Interpretation and Business Insight

- Gold price shows a persistent uptrend driven by inflation and demand
- Volatility clusters in regimes — calm periods alternate with turbulent ones
- Safe-haven spikes occur during geopolitical or economic stress
- Gold is near-random-walk territory — very difficult to forecast precisely
- Trend-following models capture the drift; mean-reversion works during spikes

## Limitations

1. Synthetic — real gold depends on USD, real rates, central bank policy, geopolitics
2. No macroeconomic features (interest rates, inflation, USD index)
3. No cross-asset correlations (gold vs equities, bonds)
4. Simplified volatility — real gold has fat tails and leverage effects
5. No intraday data or bid-ask spreads

## How to Improve This Project

1. Add macro features (US 10Y yield, DXY, CPI, VIX)
2. Model returns with GARCH for volatility forecasting
3. Include sentiment indicators (gold ETF flows, COT positioning)
4. Test regime-switching models (Markov switching)
5. Use Chronos-Bolt or TimesFM for foundation-model forecasting

## Production Considerations

- Daily price forecasting for portfolio allocation
- Risk management and hedge ratio calculation
- Integration with market data feeds (Bloomberg, Reuters)
- Automated alert systems for regime changes

## Common Mistakes

1. Expecting high accuracy — gold is near-efficient-market
2. Ignoring transaction costs when evaluating trading signals
3. Using levels instead of returns for model training
4. Not accounting for volatility regimes
5. Overfitting to historical patterns that don't persist

## Mini Challenge / Exercises

1. Compute daily returns and test for stationarity (ADF test)
2. Identify high-volatility regimes from the data
3. Build a directional forecast (up/down) and compare accuracy
4. Compare RMSE on levels vs returns — which is more useful?

## Final Summary / Key Takeaways

1. Gold price forecasting is inherently difficult (near random walk)
2. The uptrend provides drift that helps naive models
3. Volatility forecasting is often more valuable than point forecasts
4. Safe-haven spikes are driven by external events (hard to predict)
5. Real deployment needs macro features and proper risk management

In [18]:
import json
metrics = {
    'project': 'Gold Price Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Gold Price Forecasting — Complete ===")

Metrics saved.

=== Gold Price Forecasting — Complete ===
